The COVID-19 pandemic significantly shifted global daily routines. With lockdowns restricting outdoor leisure activities like visiting cafes and shopping malls, many turned to reading as their primary hobby. This sudden surge in book consumption caught the attention of startups racing to develop new digital products for book lovers.

This project analyzes a relational database from a competing platform in the book application market. The goal is to extract key data insights regarding book popularity, publisher output, author performance, and user engagement metrics. These insights will serve as the strategic foundation for a data-driven value proposition for a new digital book product.

### Conectando no banco de dados

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

db_config = {
    'user': os.getenv('DB_USER'),
    'pwd':  os.getenv('DB_PWD'),
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT'),
    'db':   os.getenv('DB_NAME')
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode': 'require'})

In [6]:
def consult_sql(query):
    return pd.io.sql.read_sql(query, con=engine)

# **Business Questions & SQL Solutions**

### **1. Market Volume: Post-2000 Book Releases**

Objective: Determine the scale of modern books available in the catalog published after January 1, 2000.

In [15]:
books_2000 = consult_sql(
    '''SELECT COUNT(title) AS book_count
    FROM books 
    WHERE publication_date > '2000-01-01';''')

display(books_2000)

,book_count
0,819


**Key Finding:** 

The dataset contains 819 books launched after 2000, indicating a rich collection of contemporary titles.

### **2. User Engagement: Review Counts and Average Ratings per Book**

Calculate the total number of textual reviews and the average score for every book to isolate top-performing content.

In [16]:
engagement_rating = consult_sql(
    '''
SELECT
    books.title,
    subquery.avg_rating,
    subquery.review_count
FROM
    (SELECT
         reviews.book_id AS book_id,
         COUNT(DISTINCT reviews.review_id) AS review_count,
         FLOOR(AVG(ratings.rating)) AS avg_rating
     FROM
         reviews
         INNER JOIN ratings ON ratings.book_id = reviews.book_id
     GROUP BY
         reviews.book_id) AS subquery
    INNER JOIN books ON subquery.book_id = books.book_id
ORDER BY
    review_count DESC,
    avg_rating DESC;
    '''
)

display(engagement_rating.head())

,title,avg_rating,review_count
0,Twilight (Twilight #1),3.0,7
1,The Glass Castle,4.0,6
2,The Book Thief,4.0,6
3,Harry Potter and the Chamber of Secrets (Harry...,4.0,6
4,Harry Potter and the Prisoner of Azkaban (Harr...,4.0,6


**Key Finding:**
Twilight emerged as the most engaging title in the dataset with 7 distinct reviews, maintaining an average floored score of 3.0.

### **3. Supply Chain Insights: Top Publishers for Full-Length Content**
Identify which publishing houses produce the highest volume of substantial books (filtering out brochures and short publications under 50 pages).

In [9]:
publisheres = consult_sql(
'''
SELECT 
    publishers.publisher,
    COUNT(publishers.publisher) AS book_count
FROM 
    publishers 
    INNER JOIN books ON publishers.publisher_id = books.publisher_id
WHERE 
    books.num_pages > 50
GROUP BY 
    publishers.publisher
ORDER BY 
    book_count DESC;
'''
)

display(publisheres.head())

,publisher,book_count
0,Penguin Books,42
1,Vintage,31
2,Grand Central Publishing,25
3,Penguin Classics,24
4,Ballantine Books,19


**Key Finding:** World-renowned authors who penned timeless, best-selling book series—such as J.K. Rowling, Rick Riordan, and J.R.R. Tolkien—dominate the top tiers of user satisfaction metrics.

### **4. High-Value Content Creators: Top-Rated Authors**
Identify the highest-rated authors, filtering exclusively for books with a high density of feedback (minimum 50 ratings) to ensure statistical reliability.

In [17]:
top_authors = consult_sql(
'''
SELECT
    authors.author,
    AVG(subquery2.avg_rating) AS final_avg            
FROM
    (SELECT
         books.title,
         books.author_id,
         subquery1.avg_rating
     FROM
         (SELECT
              book_id,
              COUNT(rating_id) AS rating_count,
              FLOOR(AVG(rating)) AS avg_rating
          FROM
              ratings
          GROUP BY
              book_id
          HAVING
              COUNT(rating_id) > 50) AS subquery1
         INNER JOIN books ON books.book_id = subquery1.book_id) AS subquery2
    INNER JOIN authors ON authors.author_id = subquery2.author_id
GROUP BY
    authors.author
ORDER BY
    final_avg DESC;
'''
)

display(top_authors)

,author,final_avg
0,J.K. Rowling/Mary GrandPré,4.0
1,Rick Riordan,4.0
2,Markus Zusak/Cao Xuân Việt Khương,4.0
3,Louisa May Alcott,4.0
4,J.R.R. Tolkien,4.0
5,George Orwell/Boris Grabnar/Peter Škerl,3.0
6,John Steinbeck,3.0
7,Lois Lowry,3.0
8,Paulo Coelho/Alan R. Clarke/Özdemir İnce,3.0
9,William Golding,3.0


**Key Findings:** World-renowned authors who penned timeless, best-selling book series—such as J.K. Rowling, Rick Riordan, and J.R.R. Tolkien—dominate the top tiers of user satisfaction metrics.

### **5. Super-User Behavior: Average Review Rates of Power Reviewers**
Calculate the average number of textual reviews generated specifically by power users (defined as users who have provided more than 50 individual book ratings).

In [18]:
avg_rating = consult_sql(
'''
SELECT
    FLOOR(AVG(subquery2.review_count)) AS avg_review_count
FROM
    (SELECT
         COUNT(reviews.review_id) AS review_count,
         subquery1.username
     FROM
         (SELECT
              username,
              COUNT(rating_id) AS rating_count
          FROM
              ratings
          GROUP BY
              username
          HAVING
              COUNT(rating_id) > 50) AS subquery1
         INNER JOIN reviews ON reviews.username = subquery1.username
     GROUP BY
         subquery1.username) AS subquery2;
'''
)

display(avg_rating.head())

,avg_review_count
0,24.0
